# RQ1 — Confidence Thresholds, Uncertainty Estimation & Selective Classification

> **Nota metodológica:** En `particiones/` no existen etiquetas clínicas gold a nivel de mensaje (solo a nivel de usuario). Siguiendo el protocolo acordado, las métricas a nivel de mensaje de este notebook usan la etiqueta heredada del usuario como **proxy débil**, sin interpretarse como etiquetas clínicas verificadas.

Métodos evaluados por trastorno (`Ansiedad`, `Depresion`, `TCA`), condición (`initial_labels`, `knn_include_same_subject`) y 5 semillas:
1. **Umbral fijo de probabilidad:** $t \in \{0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9\}$.
2. **Calibración de probabilidades (out-of-fold agrupado por `subject_id` en `dev`):** original vs Temperature Scaling vs Platt Scaling vs Isotonic Regression (ECE, Brier Score, NLL, Macro-F1 y Reliability Diagrams).
3. **Umbral de confianza:** $\text{Confidence}(x) = \max(p, 1-p)$.
4. **Incertidumbre por entropía normalizada:** $H_{\text{norm}}(p) = H(p)/\log 2$.
5. **Clasificación selectiva y abstención:** curvas **Coverage vs Selective Risk** y **AURC**.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/Urkibiris05/dm_project.git'
GIT_BRANCH = 'rq_larra'
GITHUB_TOKEN = ''

def _is_repo_root(p: Path) -> bool:
    return (p / 'particiones').is_dir() and (p / 'rq_larra' / 'rq_utils.py').is_file()

def _find_repo() -> Path | None:
    candidates = [Path('/content/dm_project'), Path.cwd().resolve(), *Path.cwd().resolve().parents]
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.is_dir():
        candidates.append(drive_root / 'dm_project')
        for pattern in ('*', '*/*', '*/*/*'):
            for p in drive_root.glob(pattern):
                if p.is_dir() and p.name == 'dm_project':
                    candidates.append(p)
    return next((p for p in candidates if _is_repo_root(p)), None)

PROJECT_ROOT = _find_repo()
if PROJECT_ROOT is None and IN_COLAB:
    clone_dir = Path('/content/dm_project')
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@') if GITHUB_TOKEN else REPO_URL
    if not clone_dir.exists():
        subprocess.run(['git', 'clone', '-b', GIT_BRANCH, clone_url, str(clone_dir)], text=True, check=False)
    PROJECT_ROOT = _find_repo()

if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró dm_project con particiones/ y rq_larra/rq_utils.py. Sube el repo a Drive o haz push a GitHub.')

if str(PROJECT_ROOT / 'rq_larra') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'rq_larra'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rq_utils

DRIVE_STORAGE_DIR = '/content/drive/MyDrive/dm_project_artifacts' if IN_COLAB else None
STORAGE_ROOT = rq_utils.resolve_storage_root(PROJECT_ROOT, DRIVE_STORAGE_DIR)

backup_base = STORAGE_ROOT / 'rq_larra_backup' / 'csv' / 'rq_baseline'
local_base = PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq_baseline'
if not (local_base / 'latest_baseline.json').is_file() and (backup_base / 'latest_baseline.json').is_file():
    shutil.copytree(backup_base, local_base, dirs_exist_ok=True)
    print('Restaurado baseline desde Drive:', backup_base)

# Auto-carga la última corrida del baseline (o especifica BASELINE_RUN_ID manualmente)
BASELINE_RUN_ID = None
pointer = rq_utils.load_latest_baseline_pointer(PROJECT_ROOT)
if BASELINE_RUN_ID is None:
    BASELINE_RUN_ID = pointer['run_id']

BASELINE_CSV_DIR = PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq_baseline' / BASELINE_RUN_ID
DEV_MSG_CSV = BASELINE_CSV_DIR / 'dev_message_predictions.csv'
if not DEV_MSG_CSV.is_file():
    raise FileNotFoundError(f'Falta {DEV_MSG_CSV}. Ejecuta primero 00_baseline_relabeling.ipynb.')

RUN_ID = rq_utils.make_run_id('rq1')
CSV_DIR = PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq1' / RUN_ID
FIG_DIR = PROJECT_ROOT / 'rq_larra' / 'figuras' / 'rq1' / RUN_ID
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

dev_msgs = pd.read_csv(DEV_MSG_CSV, dtype={'subject_id': str, 'id_message': str})
print(f'RUN_ID: {RUN_ID} | Baseline: {BASELINE_RUN_ID} | Filas dev: {len(dev_msgs)}')
dev_msgs.head()

In [ ]:
# 1) Baseline: barrido de umbral fijo de probabilidad t
PROB_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

fixed_rows = []
for (disorder, condition, seed), grp in dev_msgs.groupby(['disorder', 'condition', 'seed']):
    y = grp['label'].to_numpy(dtype=np.int64)
    p = grp['prob'].to_numpy(dtype=np.float64)
    for t in PROB_THRESHOLDS:
        pred = (p >= t).astype(np.int64)
        m = rq_utils.compute_binary_metrics(y, pred, p)
        fixed_rows.append({
            'disorder': disorder,
            'condition': condition,
            'seed': seed,
            'threshold_t': t,
            **m,
        })

fixed_df = pd.DataFrame(fixed_rows)
fixed_df.to_csv(CSV_DIR / 'fixed_threshold_seed_metrics.csv', index=False)
fixed_summary = (
    fixed_df.groupby(['disorder', 'condition', 'threshold_t'])[['macro_f1', 'precision_pos', 'recall_pos', 'fpr', 'fnr']]
    .agg(['mean', 'std']).reset_index()
)
fixed_summary.columns = ['_'.join(c).rstrip('_') for c in fixed_summary.columns]
fixed_summary.to_csv(CSV_DIR / 'fixed_threshold_summary.csv', index=False)
fixed_summary[fixed_summary['threshold_t'].isin([0.3, 0.5, 0.7])]

In [ ]:
# 2) Calibración out-of-fold agrupada por sujeto en dev (anti-leakage)
CAL_METHODS = {
    'uncalibrated': 'prob_uncal',
    'temperature': 'prob_temp',
    'platt': 'prob_platt',
    'isotonic': 'prob_isotonic',
}

cal_frames = []
cal_rows = []
full_calibrators = {}

for (disorder, condition, seed), grp in dev_msgs.groupby(['disorder', 'condition', 'seed']):
    df_cal, fitted_full = rq_utils.fit_subject_oof_calibrators(grp, n_splits=5)
    cal_frames.append(df_cal)
    full_calibrators[(disorder, condition, int(seed))] = fitted_full

    y = df_cal['label'].to_numpy(dtype=np.int64)
    for cal_name, col in CAL_METHODS.items():
        p_cal = df_cal[col].to_numpy(dtype=np.float64)
        cal_m = rq_utils.compute_calibration_metrics(y, p_cal, n_bins=10)
        cls_m = rq_utils.compute_binary_metrics(y, (p_cal >= 0.5).astype(np.int64), p_cal)
        cal_rows.append({
            'disorder': disorder,
            'condition': condition,
            'seed': seed,
            'calibrator': cal_name,
            **cal_m,
            'macro_f1_at_05': cls_m['macro_f1'],
        })

dev_cal_msgs = pd.concat(cal_frames, ignore_index=True)
cal_seed_df = pd.DataFrame(cal_rows)
cal_seed_df.to_csv(CSV_DIR / 'calibration_seed_metrics.csv', index=False)

cal_summary = (
    cal_seed_df.groupby(['disorder', 'condition', 'calibrator'])[['ece', 'brier', 'nll', 'macro_f1_at_05']]
    .agg(['mean', 'std']).reset_index()
)
cal_summary.columns = ['_'.join(c).rstrip('_') for c in cal_summary.columns]
cal_summary.to_csv(CSV_DIR / 'calibration_summary.csv', index=False)
cal_summary

In [ ]:
# 3) Reliability diagrams por trastorno (condición knn_include_same_subject, media sobre probabilidades OOF)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, disorder in zip(axes, rq_utils.DISORDERS):
    sub = dev_cal_msgs[(dev_cal_msgs['disorder'] == disorder) & (dev_cal_msgs['condition'] == 'knn_include_same_subject')]
    y = sub['label'].to_numpy(dtype=np.int64)
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfecta')
    for cal_name, col in CAL_METHODS.items():
        ece, bins_df = rq_utils.compute_ece(y, sub[col].to_numpy(dtype=np.float64), n_bins=10)
        valid = bins_df.dropna(subset=['mean_prob', 'emp_pos_rate'])
        ax.plot(valid['mean_prob'], valid['emp_pos_rate'], marker='o', label=f'{cal_name} (ECE={ece:.3f})')
    ax.set_title(f'{disorder} (k-NN include same subj)')
    ax.set_xlabel('Probabilidad predicha media')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Frecuencia empírica (proxy usuario)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'reliability_diagrams.png', dpi=150)
plt.show()

In [ ]:
# 4) Umbrales de confianza, entropía normalizada y clasificación selectiva
CONF_GRID = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
ENT_GRID = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2]

sel_rows = []
aurc_rows = []
curve_frames = []

for (disorder, condition, seed), grp in dev_cal_msgs.groupby(['disorder', 'condition', 'seed']):
    y = grp['label'].to_numpy(dtype=np.int64)
    for cal_name, col in CAL_METHODS.items():
        p_cal = grp[col].to_numpy(dtype=np.float64)
        for strat, grid in [('confidence', CONF_GRID), ('entropy', ENT_GRID)]:
            df_s = rq_utils.evaluate_selective_thresholds(y, p_cal, strategy=strat, grid=grid)
            df_s['disorder'] = disorder
            df_s['condition'] = condition
            df_s['seed'] = seed
            df_s['calibrator'] = cal_name
            sel_rows.append(df_s)

        curve_df, aurc = rq_utils.compute_risk_coverage_curve(y, p_cal, num_points=40)
        curve_df['disorder'] = disorder
        curve_df['condition'] = condition
        curve_df['seed'] = seed
        curve_df['calibrator'] = cal_name
        curve_frames.append(curve_df)
        aurc_rows.append({
            'disorder': disorder,
            'condition': condition,
            'seed': seed,
            'calibrator': cal_name,
            'aurc': aurc,
        })

selective_df = pd.concat(sel_rows, ignore_index=True)
selective_df.to_csv(CSV_DIR / 'selective_thresholds_seed.csv', index=False)

selective_summary = (
    selective_df.groupby(['disorder', 'condition', 'calibrator', 'strategy', 'threshold'])
    [['coverage', 'selective_risk', 'macro_f1', 'precision_pos', 'recall_pos', 'fpr', 'fnr']]
    .agg(['mean', 'std']).reset_index()
)
selective_summary.columns = ['_'.join(c).rstrip('_') for c in selective_summary.columns]
selective_summary.to_csv(CSV_DIR / 'selective_thresholds_summary.csv', index=False)

aurc_df = pd.DataFrame(aurc_rows)
aurc_df.to_csv(CSV_DIR / 'aurc_seed.csv', index=False)
aurc_summary = aurc_df.groupby(['disorder', 'condition', 'calibrator'])['aurc'].agg(['mean', 'std']).reset_index()
aurc_summary.to_csv(CSV_DIR / 'aurc_summary.csv', index=False)
aurc_summary

In [ ]:
# 5) Curvas Coverage vs Selective Risk y Macro-F1 vs Coverage
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
sub_sel = selective_summary[(selective_summary['strategy'] == 'confidence') & (selective_summary['calibrator'].isin(['uncalibrated', 'temperature', 'isotonic']))]

for ax, disorder in zip(axes, rq_utils.DISORDERS):
    for (condition, cal), grp in sub_sel[sub_sel['disorder'] == disorder].groupby(['condition', 'calibrator']):
        grp_s = grp.sort_values('coverage_mean')
        ls = '-' if condition == 'knn_include_same_subject' else '--'
        ax.plot(grp_s['coverage_mean'], grp_s['selective_risk_mean'], linestyle=ls, marker='.', label=f'{condition[:4]} | {cal}')
    ax.set_title(disorder)
    ax.set_xlabel('Coverage (proporción aceptada)')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes[0].set_ylabel('Selective Risk (tasa error aceptados)')
plt.suptitle('RQ1: Curvas Coverage vs Selective Risk en dev (proxy nivel mensaje)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'coverage_vs_selective_risk.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, disorder in zip(axes, rq_utils.DISORDERS):
    for (condition, cal), grp in sub_sel[sub_sel['disorder'] == disorder].groupby(['condition', 'calibrator']):
        grp_s = grp.sort_values('coverage_mean')
        ls = '-' if condition == 'knn_include_same_subject' else '--'
        ax.plot(grp_s['coverage_mean'], grp_s['macro_f1_mean'], linestyle=ls, marker='.', label=f'{condition[:4]} | {cal}')
    ax.set_title(disorder)
    ax.set_xlabel('Coverage')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes[0].set_ylabel('Macro-F1 en mensajes aceptados')
plt.suptitle('RQ1: Macro-F1 selectivo vs Coverage en dev')
plt.tight_layout()
plt.savefig(FIG_DIR / 'coverage_vs_macro_f1.png', dpi=150)
plt.show()

In [ ]:
# 6) Registro en index_runs.csv y respaldo en Drive
rq_utils.register_run(
    PROJECT_ROOT,
    run_id=RUN_ID,
    rq='rq1',
    stage='dev_analysis',
    status='completed',
    notebook='rq_larra/ipynb/rq1.ipynb',
    seed='11,23,37,51,79',
    disorder='all',
    embeddings='beto_sts_768',
    artifacts_dir=f'rq_larra/csv/rq1/{RUN_ID}',
    notes=f'baseline_run={BASELINE_RUN_ID}; OOF subject calibration + selective curves',
)
if IN_COLAB and not str(PROJECT_ROOT).startswith('/content/drive'):
    backup_root = STORAGE_ROOT / 'rq_larra_backup'
    shutil.copytree(CSV_DIR, backup_root / 'csv' / 'rq1' / RUN_ID, dirs_exist_ok=True)
    shutil.copytree(FIG_DIR, backup_root / 'figuras' / 'rq1' / RUN_ID, dirs_exist_ok=True)
    shutil.copy2(PROJECT_ROOT / 'rq_larra' / 'csv' / 'index_runs.csv', backup_root / 'csv' / 'index_runs.csv')
    print('Backup sincronizado en Drive:', backup_root)

In [ ]:
# 7) [SOLO CON PROTOCOLO CONGELADO] Evaluación final sobre test con calibradores ajustados en dev
RUN_FROZEN_TEST = False
FROZEN_CONFIDENCE_C = 0.70  # Elegido en dev antes de mirar test

if RUN_FROZEN_TEST:
    test_csv = BASELINE_CSV_DIR / 'test_message_predictions.csv'
    test_msgs = pd.read_csv(test_csv, dtype={'subject_id': str, 'id_message': str})
    test_rows = []
    for (disorder, condition, seed), grp in test_msgs.groupby(['disorder', 'condition', 'seed']):
        cals = full_calibrators[(disorder, condition, int(seed))]
        logits = grp['logit'].to_numpy(dtype=np.float64)
        probs = grp['prob'].to_numpy(dtype=np.float64)
        y = grp['label'].to_numpy(dtype=np.int64)
        p_temp = rq_utils._apply_temperature(logits, cals['temperature'])
        p_platt = cals['platt'].predict_proba(logits.reshape(-1, 1))[:, 1]
        p_iso = cals['isotonic'].predict(probs)
        for cal_name, p_vec in [('uncalibrated', probs), ('temperature', p_temp), ('platt', p_platt), ('isotonic', p_iso)]:
            cal_m = rq_utils.compute_calibration_metrics(y, p_vec)
            sel_m = rq_utils.evaluate_selective_thresholds(y, p_vec, strategy='confidence', grid=[0.5, FROZEN_CONFIDENCE_C])
            sel_m['disorder'] = disorder
            sel_m['condition'] = condition
            sel_m['seed'] = seed
            sel_m['calibrator'] = cal_name
            sel_m['ece'] = cal_m['ece']
            sel_m['brier'] = cal_m['brier']
            test_rows.append(sel_m)
    test_rq1 = pd.concat(test_rows, ignore_index=True)
    test_rq1.to_csv(CSV_DIR / 'test_frozen_rq1_metrics.csv', index=False)
    print('Evaluación congelada RQ1 guardada en test_frozen_rq1_metrics.csv')